In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from skfp.fingerprints import ECFPFingerprint
from skfp.preprocessing import MolFromSmilesTransformer

# Load splits
train_df = pd.read_parquet('../data/train.parquet')
val_df = pd.read_parquet('../data/val.parquet')
test_df = pd.read_parquet('../chebi_dataset_test_empty.parquet')
label_cols = [c for c in train_df.columns if c.startswith('class_')]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {len(label_cols)}")

Train: 26934, Val: 6734, Test: 11223, Classes: 500


In [3]:
# Compute ECFP fingerprints
mol_transformer = MolFromSmilesTransformer()
ecfp = ECFPFingerprint(fp_size=2048, radius=2, count=True)

X_train = ecfp.transform(mol_transformer.transform(train_df['SMILES'].tolist()))
X_val = ecfp.transform(mol_transformer.transform(val_df['SMILES'].tolist()))
X_test = ecfp.transform(mol_transformer.transform(test_df['SMILES'].tolist()))

y_train = train_df[label_cols].values
y_val = val_df[label_cols].values

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:15] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not removing hydrogen atom without neighbors
[14:00:16] WARNING: not r

X_train: (26934, 2048), X_val: (6734, 2048), X_test: (11223, 2048)


In [ ]:
# Train and evaluate
# rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
# rf.fit(X_train, y_train)

# y_pred = rf.predict(X_val)
# per_class_f1 = [f1_score(y_val[:, i], y_pred[:, i], zero_division=0) for i in range(len(label_cols))]
# print(f"Val Macro F1: {np.mean(per_class_f1):.4f}")

Val Macro F1: 0.4804


: 

In [4]:
# Retrain on full data (train + val)
full_df = pd.read_parquet('../chebi_dataset_train.parquet')
X_full = ecfp.transform(mol_transformer.transform(full_df['SMILES'].tolist()))
y_full = full_df[label_cols].values

rf_full = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
rf_full.fit(X_full, y_full)
print("Trained on full data")

[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] WARNING: not removing hydrogen atom without neighbors
[14:00:23] Unusual charge on atom 0 number of radical electrons set to zero
[14:00:24] WARNING: not removing hydrogen atom without neighbors
[14:00:24] WARNING: not removing hydrogen atom without neighbors
[14:00:24] WARNING: not removing hydrogen atom without neighbors
[14:00:24] WARNING: not removing hydrogen atom without neighbors
[14:00:24] WARNING: not removing hydrogen atom without neighbors
[14:00:24] WAR

Trained on full data


In [5]:
# Predict test in batches and save submission
batch_size = 1000
preds = []
for i in range(0, len(X_test), batch_size):
    preds.append(rf_full.predict(X_test[i:i+batch_size]))
y_test_pred = np.vstack(preds)

submission = pd.DataFrame(y_test_pred, columns=label_cols)
submission.insert(0, 'mol_id', test_df['mol_id'].values)
submission.insert(1, 'SMILES', test_df['SMILES'].values)
submission.to_parquet('../submissions/rf_ecfp.parquet', index=False)
print(f"Saved: {submission.shape}")

Saved: (11223, 502)
